# Parameter labels, bounds, and tying

Every parameter of a pyGSTi model carries a label. Those labels are what let you reach into a model and say something about one specific number: bound it to an interval, or tie it to another parameter so the two always move together. This page covers all three, in that order. The techniques apply to any model type, so the choice of model below is arbitrary; they're most useful when you're building your own objects (see the [custom operator tutorial](CustomOperators)) whose parameters have restrictions.

The examples below use a 1-qubit `H+s` model, whose parameterization is small enough to print in full.

In [1]:
import pygsti
import numpy as np
from pygsti.modelpacks import smq1Q_XY as std

mdl = std.target_model("H+s")

## Getting parameter labels

A `Model`'s parameters have corresponding labels, which you can get at in a few ways. Individual operators have labeled parameters too. An `OpModel` (an `ExplicitOpModel` or `ImplicitOpModel`, say) sets default parameter labels from the labels of the operators it contains, but the model's parameters can then vary independently of them.

In [2]:
# print the raw labels, straight up
mdl.parameter_labels

array([(Label('rho0'), 'X Hamiltonian error coefficient'),
       (Label('rho0'), 'Y Hamiltonian error coefficient'),
       (Label('rho0'), 'Z Hamiltonian error coefficient'),
       (Label('rho0'), 'X stochastic coefficient'),
       (Label('rho0'), 'Y stochastic coefficient'),
       (Label('rho0'), 'Z stochastic coefficient'),
       (Label('Mdefault'), 'X Hamiltonian error coefficient'),
       (Label('Mdefault'), 'Y Hamiltonian error coefficient'),
       (Label('Mdefault'), 'Z Hamiltonian error coefficient'),
       (Label('Mdefault'), 'X stochastic coefficient'),
       (Label('Mdefault'), 'Y stochastic coefficient'),
       (Label('Mdefault'), 'Z stochastic coefficient'),
       (Label(('Gxpi2', 0)), 'X Hamiltonian error coefficient'),
       (Label(('Gxpi2', 0)), 'Y Hamiltonian error coefficient'),
       (Label(('Gxpi2', 0)), 'Z Hamiltonian error coefficient'),
       (Label(('Gxpi2', 0)), 'X stochastic coefficient'),
       (Label(('Gxpi2', 0)), 'Y stochastic coefficient'),

Raw labels are usually tuples of `(op_label, description)`. You can overwrite any of them:

In [3]:
# model parameters can be set to arbitrary user-defined values
mdl.set_parameter_label(index=0, label="My favorite parameter")

In [4]:
# Model parameters in a nice format for printing
mdl.parameter_labels_pretty

['My favorite parameter',
 'rho0: Y Hamiltonian error coefficient',
 'rho0: Z Hamiltonian error coefficient',
 'rho0: X stochastic coefficient',
 'rho0: Y stochastic coefficient',
 'rho0: Z stochastic coefficient',
 'Mdefault: X Hamiltonian error coefficient',
 'Mdefault: Y Hamiltonian error coefficient',
 'Mdefault: Z Hamiltonian error coefficient',
 'Mdefault: X stochastic coefficient',
 'Mdefault: Y stochastic coefficient',
 'Mdefault: Z stochastic coefficient',
 'Gxpi2:0: X Hamiltonian error coefficient',
 'Gxpi2:0: Y Hamiltonian error coefficient',
 'Gxpi2:0: Z Hamiltonian error coefficient',
 'Gxpi2:0: X stochastic coefficient',
 'Gxpi2:0: Y stochastic coefficient',
 'Gxpi2:0: Z stochastic coefficient',
 'Gypi2:0: X Hamiltonian error coefficient',
 'Gypi2:0: Y Hamiltonian error coefficient',
 'Gypi2:0: Z Hamiltonian error coefficient',
 'Gypi2:0: X stochastic coefficient',
 'Gypi2:0: Y stochastic coefficient',
 'Gypi2:0: Z stochastic coefficient']

The "pretty" form flattens each tuple into a single string, joining the pieces with `": "`. Either form works where a parameter label is expected, but don't mix them inside one call. `collect_parameters`, for example, resolves its entire list against `parameter_labels` and only falls back to `parameter_labels_pretty` wholesale, so a list mixing raw tuples with pretty strings raises `KeyError`. The same goes for mixing integer indices with labels of either kind.

In [5]:
# For a single operator: you can get its "local" parameter labels
# (in general different from the model's parameter labels)
mdl.operations[('Gxpi2',0)].parameter_labels

array(['X Hamiltonian error coefficient',
       'Y Hamiltonian error coefficient',
       'Z Hamiltonian error coefficient', 'X stochastic coefficient',
       'Y stochastic coefficient', 'Z stochastic coefficient'],
      dtype=object)

In [6]:
# The parameters of all the operators, with mappings to non-default model parameters
mdl.print_parameters_by_op()

*** MODEL PARAMETERS (24 total) ***
>>> rho0 [ComposedState]: 6 params, indices=slice(np.int64(0), np.int64(6), None), 2 sub-members
   0: X Hamiltonian error coefficient --> My favorite parameter
   1: Y Hamiltonian error coefficient
   2: Z Hamiltonian error coefficient
   3: X stochastic coefficient
   4: Y stochastic coefficient
   5: Z stochastic coefficient
>>> Mdefault [ComposedPOVM]: 6 params, indices=slice(np.int64(6), np.int64(12), None), 2 sub-members
   6: X Hamiltonian error coefficient
   7: Y Hamiltonian error coefficient
   8: Z Hamiltonian error coefficient
   9: X stochastic coefficient
   10: Y stochastic coefficient
   11: Z stochastic coefficient
>>> Gxpi2:0 [ComposedOp]: 6 params, indices=slice(np.int64(12), np.int64(18), None), 2 sub-members
   12: X Hamiltonian error coefficient
   13: Y Hamiltonian error coefficient
   14: Z Hamiltonian error coefficient
   15: X stochastic coefficient
   16: Y stochastic coefficient
   17: Z stochastic coefficient
>>> Gypi2:0 

## Bounding a parameter's values

Optimizers respect per-parameter bounds, so a bound is the way to keep a fit from wandering into values you know are unphysical or uninteresting. Suppose you want to hold the coherent Z error on the $X(\pi/2)$ gate between 0 and 0.2. Bounds that survive a rebuild live on model members rather than on the model itself (you can set them on the model, but see the caveat at the end of the next section), so first find the member that owns the parameter.

In [7]:
# Here's the X(pi/2) gate:
print(mdl.operations[('Gxpi2', 0)])

Composed operation of 2 factors:
Factor 0:
<pygsti.modelmembers.operations.staticunitaryop.StaticUnitaryOp object at 0xADDRESS>Factor 1:
Exponentiated operation map with dim = 4, num params = 6



In [8]:
# this is the error generator whose parameters we want to bound
eg = mdl.operations[('Gxpi2', 0)].factorops[1].errorgen
for i, lbl in enumerate(eg.parameter_labels):
    print(i, lbl)

0 X Hamiltonian error coefficient
1 Y Hamiltonian error coefficient
2 Z Hamiltonian error coefficient
3 X stochastic coefficient
4 Y stochastic coefficient
5 Z stochastic coefficient


The parameter we want has index 2. Currently the bounds are `None`, which means there aren't any:

In [9]:
print(eg.parameter_bounds)

None


Set the `parameter_bounds` attribute of a model member to a 2D NumPy array of shape `(num_params, 2)`, whose rows are `(min, max)` for each parameter. Use `numpy.inf` and `-numpy.inf` where you don't want one or both bounds.

In [10]:
bounds = np.empty((eg.num_params, 2), 'd')
bounds[:, 0] = -np.inf  # initial lower bounds
bounds[:, 1] = np.inf   # initial upper bounds
bounds[2, :] = (0, 0.2) # bounds for "Z Hamiltonian error coefficient" parameter
eg.parameter_bounds = bounds

Setting bounds on a member marks the containing model as needing a rebuild, but you don't have to trigger that rebuild yourself. Reading `mdl.parameter_bounds` cleans up the model's parameter vector first, so the member's bounds are already in place the first time you look. The model's bounds array has one row per model parameter, and only the row belonging to our error coefficient is finite:

In [11]:
model_bounds = mdl.parameter_bounds
bounded = np.flatnonzero(np.isfinite(model_bounds).any(axis=1))
for i in bounded:
    print(i, mdl.parameter_labels_pretty[i], model_bounds[i])

14 Gxpi2:0: Z Hamiltonian error coefficient [0.  0.2]


From here on, an optimization of `mdl` will keep that parameter inside the interval.

## Tying parameters together

`collect_parameters` replaces several parameters with a single one, so that everything that used to read the originals now reads the same number. This is how you impose "these errors are equal" without writing a new operator class.

The next cell prints a warning about model-level parameter bounds being overwritten. It fires on any rebuild where the model is holding a bounds array, and the model is holding one here only because reading `mdl.parameter_bounds` above cached the member's bounds onto it. Nothing is lost in that case, since the rebuild re-reads the same bounds off the member. The warning is worth heeding when you set bounds on the model itself; the last paragraph of this section says why.

In [12]:
mdl.collect_parameters([ (('Gxpi2',0), 'X Hamiltonian error coefficient'),
                         (('Gypi2',0), 'Y Hamiltonian error coefficient')],
                       new_param_label='Over-rotation')

<repo>/pygsti/models/model.py:950: UserWarning: Internal Model attributes are being rebuilt. This is likely because a modelmember has been either added or removed. If you have manually set parameter bounds values at the Model level (not the model member level), for example using the `set_parameter_bounds` method, these values will be overwritten by the parameter bounds found in each of the modelmembers.
  _warnings.warn(msg)


In [13]:
# Using "pretty" labels works too:
mdl.collect_parameters(['Gxpi2:0: Y stochastic coefficient',
                        'Gxpi2:0: Z stochastic coefficient' ],
                       new_param_label='Gxpi2 off-axis stochastic')

In [14]:
# You can also use integer indices, and parameter labels can be tuples too.
mdl.collect_parameters([3,4,5], new_param_label=("rho0", "common stochastic coefficient"))

Each call shrinks the parameter vector:

In [15]:
# There are now fewer parameters
mdl.parameter_labels_pretty

['My favorite parameter',
 'rho0: Y Hamiltonian error coefficient',
 'rho0: Z Hamiltonian error coefficient',
 'rho0: common stochastic coefficient',
 'Mdefault: X Hamiltonian error coefficient',
 'Mdefault: Y Hamiltonian error coefficient',
 'Mdefault: Z Hamiltonian error coefficient',
 'Mdefault: X stochastic coefficient',
 'Mdefault: Y stochastic coefficient',
 'Mdefault: Z stochastic coefficient',
 'Over-rotation',
 'Gxpi2:0: Y Hamiltonian error coefficient',
 'Gxpi2:0: Z Hamiltonian error coefficient',
 'Gxpi2:0: X stochastic coefficient',
 'Gxpi2 off-axis stochastic',
 'Gypi2:0: X Hamiltonian error coefficient',
 'Gypi2:0: Z Hamiltonian error coefficient',
 'Gypi2:0: X stochastic coefficient',
 'Gypi2:0: Y stochastic coefficient',
 'Gypi2:0: Z stochastic coefficient']

In [16]:
# And you can see how they're wired up for each op:
mdl.print_parameters_by_op()

*** MODEL PARAMETERS (20 total) ***
>>> rho0 [ComposedState]: 6 params, indices=[0 1 2 3 3 3], 2 sub-members
   0: X Hamiltonian error coefficient --> My favorite parameter
   1: Y Hamiltonian error coefficient
   2: Z Hamiltonian error coefficient
   3: X stochastic coefficient --> ('rho0', 'common stochastic coefficient')
   3: Y stochastic coefficient --> ('rho0', 'common stochastic coefficient')
   3: Z stochastic coefficient --> ('rho0', 'common stochastic coefficient')
>>> Mdefault [ComposedPOVM]: 6 params, indices=slice(np.int64(4), np.int64(10), None), 2 sub-members
   4: X Hamiltonian error coefficient
   5: Y Hamiltonian error coefficient
   6: Z Hamiltonian error coefficient
   7: X stochastic coefficient
   8: Y stochastic coefficient
   9: Z stochastic coefficient
>>> Gxpi2:0 [ComposedOp]: 6 params, indices=[10 11 12 13 14 14], 2 sub-members
   10: X Hamiltonian error coefficient --> Over-rotation
   11: Y Hamiltonian error coefficient
   12: Z Hamiltonian error coefficien

One caveat worth taking seriously: after a `collect_parameters` call, treat the model's parameter vector (the result of `to_vector()`) as having an entirely new format. Untouched parameters do not keep their old indices, so any index you cached beforehand is stale. The bounds set above survive, because they were set on the member and get re-propagated on every rebuild. Bounds set directly at the model level with `Model.set_parameter_bounds` do not survive; a rebuild overwrites them with whatever the members say.

## Un-tying parameters

The reverse operation promotes each use of a shared parameter back to an independent one. The new parameters get their labels from the operations that use them.

In [17]:
mdl.uncollect_parameters('Gxpi2 off-axis stochastic')

In [18]:
mdl.print_parameters_by_op()

*** MODEL PARAMETERS (21 total) ***
>>> rho0 [ComposedState]: 6 params, indices=[0 1 2 3 3 3], 2 sub-members
   0: X Hamiltonian error coefficient --> My favorite parameter
   1: Y Hamiltonian error coefficient
   2: Z Hamiltonian error coefficient
   3: X stochastic coefficient --> ('rho0', 'common stochastic coefficient')
   3: Y stochastic coefficient --> ('rho0', 'common stochastic coefficient')
   3: Z stochastic coefficient --> ('rho0', 'common stochastic coefficient')
>>> Mdefault [ComposedPOVM]: 6 params, indices=slice(np.int64(4), np.int64(10), None), 2 sub-members
   4: X Hamiltonian error coefficient
   5: Y Hamiltonian error coefficient
   6: Z Hamiltonian error coefficient
   7: X stochastic coefficient
   8: Y stochastic coefficient
   9: Z stochastic coefficient
>>> Gxpi2:0 [ComposedOp]: 6 params, indices=[10 11 12 13 14 20], 2 sub-members
   10: X Hamiltonian error coefficient --> Over-rotation
   11: Y Hamiltonian error coefficient
   12: Z Hamiltonian error coefficien

This is not an undo. `uncollect_parameters` splits apart every occurrence of the parameter, including ones that were shared before you ever called `collect_parameters`, and the resulting indices differ from the ones you started with.